In [ ]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.9 MB/s eta 0:00:00


## Cell 1 — Imports


In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

np.random.seed(123)

## Cell 2 — Load & clean data


Synthetic Data Creation

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# --- Generate events.csv ---
event_titles = [
    "Air Show 2026", "Macrofinance Summit", "Python for Finance Workshop",
    "Startup Networking Night", "Equity Research Webinar", "Blockchain Expo 2026",
    "Data Science Bootcamp", "Angel Investor Meetup", "Cloud Computing Summit",
    "Fintech Hackathon", "Machine Learning Conference", "Real Estate Investing Talk",
    "Cybersecurity Workshop", "Product Management Masterclass", "UX Design Sprint",
    "Crypto Trading Seminar", "AI in Healthcare Forum", "Marketing Analytics Day",
    "Leadership & Growth Summit", "DevOps Fundamentals Workshop",
    "Renewable Energy Expo", "Options Trading Bootcamp", "Web3 Developer Day",
    "HR Tech Conference", "Social Media Strategy Workshop", "Quant Finance Meetup",
    "Deep Learning Symposium", "Private Equity Forum", "Agile Scrum Masterclass",
    "Open Source Contributor Day"
]

category_pool = [
    "Finance|Conference|Economics", "Tech|Finance|Education", "Networking|Business",
    "Finance|Online|Education", "Tech|Blockchain|Expo", "Tech|Education|Bootcamp",
    "Investing|Networking|Business", "Tech|Cloud|Conference", "Tech|Finance|Hackathon",
    "Tech|AI|Conference", "Finance|Investing|Talk", "Tech|Security|Workshop",
    "Business|Education|Masterclass", "Design|Tech|Workshop", "Finance|Crypto|Seminar",
    "Health|AI|Forum", "Marketing|Analytics|Conference", "Leadership|Business|Summit",
    "Tech|DevOps|Workshop", "Energy|Outdoor|Exhibition", "Finance|Trading|Bootcamp",
    "Tech|Web3|Developer", "HR|Tech|Conference", "Marketing|Social|Workshop",
    "Finance|Quant|Meetup", "Tech|AI|Symposium", "Finance|PE|Forum",
    "Tech|Agile|Masterclass", "Tech|OpenSource|Community", "Aviation|Outdoor|Exhibition"
]

num_events = 30
events_df = pd.DataFrame({
    'eventId': range(1, num_events + 1),
    'title': event_titles[:num_events],
    'categories': category_pool[:num_events]
})

events_df.to_csv('./events.csv', index=False)
print(f"Generated events.csv with {len(events_df)} events")

# --- Generate event_interactions.csv ---
num_users = 200
base_time = datetime(2024, 1, 1)

rows = []
for user_id in range(1, num_users + 1):
    # Each user attends between 5 and 20 events
    num_attended = np.random.randint(5, 20)
    attended_events = np.random.choice(range(1, num_events + 1), size=num_attended, replace=False)

    for event_id in attended_events:
        timestamp = base_time + timedelta(days=int(np.random.randint(0, 365)))
        rows.append({
            'userId': user_id,
            'eventId': int(event_id),
            'rsvp_status': 1,
            'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
        })

interactions_df = pd.DataFrame(rows)
interactions_df.to_csv('./event_interactions.csv', index=False)
print(f"Generated event_interactions.csv with {len(interactions_df)} rows from {num_users} users")
print(interactions_df.head())

Generated events.csv with 30 events
Generated event_interactions.csv with 2339 rows from 200 users
   userId  eventId  rsvp_status            timestamp
0       1       28            1  2024-06-18 00:00:00
1       1       16            1  2024-07-06 00:00:00
2       1       24            1  2024-09-27 00:00:00
3       1       18            1  2024-07-08 00:00:00
4       1        9            1  2024-06-23 00:00:00


CONT....

In [ ]:
interactions = pd.read_csv('./event_interactions.csv')
interactions['timestamp'] = pd.to_datetime(interactions['timestamp'])

unique_users = interactions['userId'].unique()
if len(unique_users) > 50:
    rand_userIds = np.random.choice(unique_users, size=int(len(unique_users)*0.3), replace=False)
    interactions = interactions.loc[interactions['userId'].isin(rand_userIds)]
else:
    print("Small dataset detected. Using all available users.")
    rand_userIds = unique_users

print('There are {} rows of data from {} users'.format(len(interactions), len(rand_userIds)))

There are 700 rows of data from 60 users


## Cell 3 — Encode IDs (new — prevents embedding index errors)


In [ ]:
user_encoder = LabelEncoder()
event_encoder = LabelEncoder()

interactions['userId'] = user_encoder.fit_transform(interactions['userId'])
interactions['eventId'] = event_encoder.fit_transform(interactions['eventId'])

num_users = interactions['userId'].nunique()
num_items = interactions['eventId'].nunique()
all_eventIds = interactions['eventId'].unique()

print(f"Users: {num_users} | Events: {num_items}")

Users: 60 | Events: 30


In [ ]:
raw_user_id = 1  # since synthetic data has users 1-200
user_id = user_encoder.transform([raw_user_id])[0]

attended = set(interactions[interactions['userId'] == user_id]['eventId'].tolist())
candidates = [e for e in all_eventIds if e not in attended]

model.eval()
with torch.no_grad():
    user_tensor = torch.tensor([user_id] * len(candidates)).long()
    item_tensor = torch.tensor(candidates).long()
    scores = model(user_tensor, item_tensor).squeeze().numpy()

if scores.ndim == 0:
    scores = np.array([scores])

top_events_encoded = [candidates[i] for i in np.argsort(scores)[::-1][:10]]
top_events_original = event_encoder.inverse_transform(top_events_encoded)

events_metadata = pd.read_csv('./events.csv')
print(f"\nTop 10 recommended events for User {raw_user_id}:")
print(events_metadata[events_metadata['eventId'].isin(top_events_original)][['title', 'categories']].to_string(index=False))


Top 10 recommended events for User 1:
                         title                     categories
       Equity Research Webinar           Tech|Blockchain|Expo
         Data Science Bootcamp  Investing|Networking|Business
         Angel Investor Meetup          Tech|Cloud|Conference
        Cybersecurity Workshop Business|Education|Masterclass
Product Management Masterclass           Design|Tech|Workshop
        Crypto Trading Seminar                Health|AI|Forum
       Marketing Analytics Day     Leadership|Business|Summit
            Web3 Developer Day             HR|Tech|Conference
       Agile Scrum Masterclass      Tech|OpenSource|Community


## Cell 4 — Train/test split


In [ ]:
interactions['rank_latest'] = interactions.groupby(['userId'])['timestamp'] \
                                .rank(method='first', ascending=False)

train_interactions = interactions[interactions['rank_latest'] != 1].copy()
test_interactions  = interactions[interactions['rank_latest'] == 1].copy()

train_interactions = train_interactions[['userId', 'eventId', 'rsvp_status']]
test_interactions  = test_interactions[['userId', 'eventId', 'rsvp_status']]

train_interactions.loc[:, 'rsvp_status'] = 1
print(f"Train: {len(train_interactions)} | Test: {len(test_interactions)}")

Train: 640 | Test: 60


## Cell 5 — Dataset class


In [ ]:
class EventTrainDataset(Dataset):
    def __init__(self, interactions, all_eventIds):
        self.users, self.items, self.labels = self.get_dataset(interactions, all_eventIds)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

    def get_dataset(self, interactions, all_eventIds):
        users, items, labels = [], [], []
        user_item_set = set(zip(interactions['userId'], interactions['eventId']))
        num_negatives = 4
        for u, i in user_item_set:
            users.append(u)
            items.append(i)
            labels.append(1)
            for _ in range(num_negatives):
                negative_item = np.random.choice(all_eventIds)
                while (u, negative_item) in user_item_set:
                    negative_item = np.random.choice(all_eventIds)
                users.append(u)
                items.append(negative_item)
                labels.append(0)
        return torch.tensor(users).long(), torch.tensor(items).long(), torch.tensor(labels)

## Cell 6 — Improved model


In [ ]:
class NCF(pl.LightningModule):
    def __init__(self, num_users, num_items, interactions, all_eventIds):
        super().__init__()
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=32)
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=32)
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.2)
        self.interactions = interactions
        self.all_eventIds = all_eventIds

    def forward(self, user_input, item_input):
        user_embedded = self.user_embedding(user_input)
        item_embedded = self.item_embedding(item_input)
        vector = torch.cat([user_embedded, item_embedded], dim=-1)
        vector = self.dropout(nn.ReLU()(self.fc1(vector)))
        vector = self.dropout(nn.ReLU()(self.fc2(vector)))
        vector = nn.ReLU()(self.fc3(vector))
        return nn.Sigmoid()(self.output(vector))

    def training_step(self, batch, batch_idx):
        user_input, item_input, labels = batch
        predicted_labels = self(user_input, item_input)
        loss = nn.BCELoss()(predicted_labels, labels.view(-1, 1).float())
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3, weight_decay=1e-5)

    def train_dataloader(self):
        return DataLoader(EventTrainDataset(self.interactions, self.all_eventIds),
                          batch_size=512, num_workers=0)

## Cell 7 — Train


In [ ]:
model = NCF(num_users, num_items, train_interactions, all_eventIds)
trainer = pl.Trainer(max_epochs=20)
trainer.fit(model)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ user_embedding │ Embedding │  1.9 K │ train │     0 │
│ 1 │ item_embedding │ Embedding │    960 │ train │     0 │
│ 2 │ fc1            │ Linear    │  8.3 K │ train │     0 │
│ 3 │ fc2            │ Linear    │  8.3 K │ train │     0 │
│ 4 │ fc3            │ Linear    │  2.1 K │ train │     0 │
│ 5 │ output         │ Linear    │     33 │ train │     0 │
│ 6 │ dropout        │ Dropout   │      0 │ train │     0 │
└───┴────────────────┴───────────┴────────┴───────┴───────┘

Trainable params: 21.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.6 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


## Cell 8 — Evaluate

In [ ]:
test_user_item_set = set(zip(test_interactions['userId'], test_interactions['eventId']))
user_interacted_items = interactions.groupby('userId')['eventId'].apply(list).to_dict()

model.eval()
hits = []
for (u, i) in tqdm(test_user_item_set):
    interacted_items = user_interacted_items[u]
    not_interacted_items = set(all_eventIds) - set(interacted_items)
    num_to_sample = min(len(not_interacted_items), 99)
    selected_not_interacted = list(np.random.choice(list(not_interacted_items), num_to_sample, replace=False))
    test_items = selected_not_interacted + [i]

    with torch.no_grad():
        u_tensor = torch.tensor([u] * len(test_items)).long()
        i_tensor = torch.tensor(test_items).long()
        predicted_labels = np.squeeze(model(u_tensor, i_tensor).detach().numpy())

    top10_items = [test_items[idx] for idx in np.argsort(predicted_labels)[::-1][:10]]
    hits.append(1 if i in top10_items else 0)

print("Hit Ratio @ 10: {:.2f}".format(np.average(hits)))

  0%|          | 0/60 [00:00<?, ?it/s]

Hit Ratio @ 10: 0.53


## Cell 9 — Generate recommendations [[FINALLY RUN THIS!]]


In [ ]:
# Change this to any userId from your dataset (before encoding)
raw_user_id = int(input("Enter user id please: "))
user_id = user_encoder.transform([raw_user_id])[0]

attended  = set(interactions[interactions['userId'] == user_id]['eventId'].tolist())
candidates = [e for e in all_eventIds if e not in attended]

model.eval()
with torch.no_grad():
    user_tensor = torch.tensor([user_id] * len(candidates)).long()
    item_tensor = torch.tensor(candidates).long()
    scores = model(user_tensor, item_tensor).squeeze().numpy()

if scores.ndim == 0:
    scores = np.array([scores])

top_events_encoded = [candidates[i] for i in np.argsort(scores)[::-1][:10]]

# Decode back to original event IDs
top_events_original = event_encoder.inverse_transform(top_events_encoded)

events_metadata = pd.read_csv('./events.csv')
print(f"\nTop 10 recommended events for User {raw_user_id}:")
print(events_metadata[events_metadata['eventId'].isin(top_events_original)][['title', 'categories']])

Enter user id please: 5

Top 10 recommended events for User 5:
                             title                      categories
0                    Air Show 2026    Finance|Conference|Economics
1              Macrofinance Summit          Tech|Finance|Education
4          Equity Research Webinar            Tech|Blockchain|Expo
8           Cloud Computing Summit          Tech|Finance|Hackathon
11      Real Estate Investing Talk          Tech|Security|Workshop
13  Product Management Masterclass            Design|Tech|Workshop
16          AI in Healthcare Forum  Marketing|Analytics|Conference
17         Marketing Analytics Day      Leadership|Business|Summit
22              Web3 Developer Day              HR|Tech|Conference
26         Deep Learning Symposium                Finance|PE|Forum


In [ ]:
# Use 0 since that's the only encoded user ID
raw_user_id = user_encoder.inverse_transform([0])[0]
user_id = 0

attended = set(interactions[interactions['userId'] == user_id]['eventId'].tolist())
candidates = [e for e in all_eventIds if e not in attended]

print(f"User attended {len(attended)} events, {len(candidates)} candidates remaining")

User attended 16 events, 14 candidates remaining


# PLANNER ALGO [[RUN THIS!]]



In [ ]:
# ── Advisor logic ─────────────────────────────────────────────────
def get_ai_event_advisor(user_id, recommendations_df, budget, preferred_categories=None,
                          max_events=3, additional_notes="", location_preference="any",
                          group_size=1, experience_level="intermediate"):

    category_price_map = {
        "Finance": 80, "Tech": 60, "Networking": 40, "Education": 50,
        "AI": 90, "Bootcamp": 120, "Conference": 100, "Workshop": 70,
        "Online": 20, "Hackathon": 30, "Seminar": 50, "Expo": 45,
        "Meetup": 25, "Aviation": 60, "Health": 55, "Marketing": 65,
        "Leadership": 85, "DevOps": 70, "Energy": 45, "Trading": 95,
        "Web3": 75, "HR": 55, "Social": 35, "Quant": 110, "PE": 130,
        "Agile": 65, "OpenSource": 15, "Business": 60, "Design": 55,
        "Investing": 85, "Security": 80, "Crypto": 90,
    }
    level_map = {
        "beginner":     ["Networking", "Meetup", "Workshop", "Education", "Social", "Expo"],
        "intermediate": ["Conference", "Seminar", "Bootcamp", "Hackathon", "Marketing",
                         "Agile", "DevOps", "Design", "HR", "Leadership"],
        "advanced":     ["Finance", "AI", "Quant", "PE", "Crypto", "Trading",
                         "Web3", "Security", "Tech", "Investing"],
    }
    online_categories   = ["Online", "Education", "Seminar", "Webinar", "OpenSource"]
    inperson_categories = ["Networking", "Expo", "Meetup", "Conference",
                            "Hackathon", "Bootcamp", "Workshop", "Aviation"]

    results = []
    for _, row in recommendations_df.iterrows():
        cat_list   = [c.strip() for c in row['categories'].split('|')]
        base_price = max([category_price_map.get(c, 50) for c in cat_list])

        # ── Group pricing ─────────────────────────────────────────
        if group_size >= 3:
            group_multiplier = 0.85
            group_note = f"Group discount 15% off ({group_size} people)"
        elif group_size == 2:
            group_multiplier = 0.95
            group_note = "Pair discount 5% off"
        else:
            group_multiplier = 1.0
            group_note = "Solo attendance"

        estimated_price = round(base_price * group_size * group_multiplier)

        # ── ROI ───────────────────────────────────────────────────
        value_cats = ["AI", "Finance", "Quant", "Tech", "Security", "Trading"]
        base_roi   = sum([15 for c in cat_list if c in value_cats])
        roi_score  = round((base_roi / base_price) * 100, 1) if base_price > 0 else 0

        # ── Model confidence score (NEW) ──────────────────────────
        # Scale 0-1 model score to 0-50 points so it meaningfully
        # influences ranking without dominating other factors
        model_confidence    = float(row['model_score'])
        model_score_contrib = round(model_confidence * 50, 1)

        # ── Location ──────────────────────────────────────────────
        is_online   = any(c in online_categories   for c in cat_list)
        is_inperson = any(c in inperson_categories for c in cat_list)
        if location_preference == "online":
            location_score = 30 if is_online else (-20 if is_inperson else 0)
            location_tag = "Online" if is_online else "In-person"
        elif location_preference == "inperson":
            location_score = 30 if is_inperson else (-20 if is_online else 0)
            location_tag = "In-person" if is_inperson else "Online"
        else:
            location_score = 10
            location_tag = "Online" if is_online else "In-person"

        # ── Experience level ──────────────────────────────────────
        suitable_cats = level_map.get(experience_level, [])
        level_matches = [c for c in cat_list if c in suitable_cats]
        level_score = len(level_matches) * 25
        level_tag = "✓ Matches your level" if level_matches else "⚠ May not match your level"

        # ── Category preference ───────────────────────────────────
        category_score = len([c for c in cat_list if any(
            pref.lower() in c.lower() for pref in preferred_categories)]) * 30 \
            if preferred_categories else 20

        # ── Budget ────────────────────────────────────────────────
        fits_budget = estimated_price <= budget
        budget_score = 30 if fits_budget else -9999

        # ── Total score now includes model confidence ─────────────
        total_score = (category_score + budget_score + location_score +
                       level_score + roi_score + model_score_contrib)

        results.append({
            'title'           : row['title'],
            'categories'      : row['categories'],
            'cat_list'        : cat_list,
            'estimated_price' : estimated_price,
            'roi_score'       : roi_score,
            'model_confidence': round(model_confidence * 100, 1),  # show as %
            'location_tag'    : location_tag,
            'level_tag'       : level_tag,
            'group_note'      : group_note,
            'fits_budget'     : fits_budget,
            'score'           : total_score,
        })

    results_df = pd.DataFrame(results).sort_values('score', ascending=False)
    affordable  = results_df[results_df['fits_budget']]

    # ── Category diversity ────────────────────────────────────────
    seen_cats, diverse = set(), []
    for _, row in affordable.iterrows():
        if row['cat_list'][0] not in seen_cats:
            diverse.append(row)
            seen_cats.add(row['cat_list'][0])
        if len(diverse) == max_events:
            break

    top         = pd.DataFrame(diverse)
    total_spend = top['estimated_price'].sum()
    waitlist_msg = ""

    if total_spend > budget and len(top) > 1:
        drop_idx = top['roi_score'].idxmin()
        drop_event = top.loc[drop_idx, 'title']
        top = top.drop(drop_idx)
        total_spend = top['estimated_price'].sum()
        waitlist_msg = f"\n💡 '{drop_event}' was removed — lowest ROI for your budget."

    # ── Print ─────────────────────────────────────────────────────
    print(f"\n🎯  Event Recommendations for User {user_id}")
    print(f"    Budget: ${budget}  |  Group: {group_size}  |  "
          f"Level: {experience_level.capitalize()}  |  Location: {location_preference.capitalize()}")
    if additional_notes:
        print(f"    Notes: {additional_notes}")
    print("─" * 60)
    for i, (_, row) in enumerate(top.iterrows(), 1):
        print(f"\n{i}. {row['title']}")
        print(f"   Categories      : {row['categories']}")
        print(f"   Estimated cost  : ${row['estimated_price']}  ({row['group_note']})")
        print(f"   Model confidence: {row['model_confidence']}%")
        print(f"   ROI Score       : {row['roi_score']}  |  {row['location_tag']}  |  {row['level_tag']}")
    print("\n" + "─" * 60)
    print(f"   Total spend      : ${total_spend}")
    print(f"   Remaining budget : ${budget - total_spend}")
    if waitlist_msg: print(waitlist_msg)
    if len(top) < max_events:
        print(f"\n⚠️  Only {len(top)} events matched all your filters.")
    return top



# ── Interactive input ─────────────────────────────────────────────
print("=" * 60)
print("         🎯 EVENT RECOMMENDATION ADVISOR")
print("=" * 60)

raw_user_id = int(input("\nEnter User ID: "))
budget = int(input("Enter budget ($): "))
max_events = int(input("Max events to attend: "))
group_size = int(input("Group size (1 = solo): "))
location_preference = input("Location preference (any / online / inperson): ").strip().lower()
experience_level = input("Experience level (beginner / intermediate / advanced): ").strip().lower()

print("\nAvailable categories: Finance, Tech, Networking, AI, Education,")
print("Bootcamp, Conference, Workshop, Hackathon, Seminar, Expo,")
print("Meetup, Trading, Security, Marketing, Leadership, Design")
cats_raw = input("Preferred categories (comma separated, or Enter for any): ").strip()
preferred_categories = [c.strip() for c in cats_raw.split(",")] if cats_raw else None
additional_notes = input("Any additional notes (or press Enter to skip): ").strip()

# ── Get model recommendations + scores ───────────────────────────
try:
    user_id_enc = user_encoder.transform([raw_user_id])[0]
except Exception:
    print(f"\n❌ User ID {raw_user_id} not found. Available IDs: {list(user_encoder.classes_[:10])}...")
    raise SystemExit

attended   = set(interactions[interactions['userId'] == user_id_enc]['eventId'].tolist())
candidates = [e for e in all_eventIds if e not in attended]

if not candidates:
    print(f"\n❌ User {raw_user_id} has attended all events — no candidates left.")
    raise SystemExit

model.eval()
with torch.no_grad():
    u_tensor = torch.tensor([user_id_enc] * len(candidates)).long()
    i_tensor = torch.tensor(candidates).long()
    scores   = model(u_tensor, i_tensor).squeeze().numpy()

if scores.ndim == 0:
    scores = np.array([scores])

# ── Build recommendations df WITH model scores attached ──────────
top_indices = np.argsort(scores)[::-1][:10]
top_encoded = [candidates[i] for i in top_indices]
top_scores = [scores[i] for i in top_indices]         # <-- actual model scores
top_original = event_encoder.inverse_transform(top_encoded)

recs_df = events_metadata[events_metadata['eventId'].isin(top_original)][['eventId', 'title', 'categories']].copy()

# Map scores back onto the dataframe by eventId
score_map = dict(zip(top_original, top_scores))
recs_df['model_score'] = recs_df['eventId'].map(score_map)

get_ai_event_advisor(
    user_id = raw_user_id,
    recommendations_df = recs_df,
    budget = budget,
    preferred_categories = preferred_categories,
    max_events = max_events,
    additional_notes = additional_notes,
    location_preference = location_preference,
    group_size = group_size,
    experience_level = experience_level,
)

         🎯 EVENT RECOMMENDATION ADVISOR

Enter User ID: 2
Enter budget ($): 150
Max events to attend: 2
Group size (1 = solo): 1
Location preference (any / online / inperson): any
Experience level (beginner / intermediate / advanced): intermediate

Available categories: Finance, Tech, Networking, AI, Education,
Bootcamp, Conference, Workshop, Hackathon, Seminar, Expo,
Meetup, Trading, Security, Marketing, Leadership, Design
Preferred categories (comma separated, or Enter for any): Tech
Any additional notes (or press Enter to skip): 

🎯  Event Recommendations for User 2
    Budget: $150  |  Group: 1  |  Level: Intermediate  |  Location: Any
────────────────────────────────────────────────────────────

1. Product Management Masterclass
   Categories      : Design|Tech|Workshop
   Estimated cost  : $70  (Solo attendance)
   Model confidence: 23.8%
   ROI Score       : 21.4  |  In-person  |  ✓ Matches your level

────────────────────────────────────────────────────────────
   Total spend  

,title,categories,cat_list,estimated_price,roi_score,model_confidence,location_tag,level_tag,group_note,fits_budget,score
3,Product Management Masterclass,Design|Tech|Workshop,"[Design, Tech, Workshop]",70,21.4,23.8,In-person,✓ Matches your level,Solo attendance,True,128.3
